# Stage 6: Interactive Demo & Submission

> **Runtime Prerequisite:** This notebook requires **Stages 3-4** to have been executed
> so that the ChromaDB vector store and `processed_holdings.json` are available.

## Showcase Architecture
| Component | Implementation |
|---|---|
| **Retrieval** | EnsembleRetriever (BM25 + ChromaDB Vector) |
| **Reranking** | FlashrankRerank (narrow=8, wide=20) |
| **Synthesis** | Routed Stuff / Map-Reduce chain |
| **Filters** | Entity-scoped + Temporal + Fund-diversity |

## Thematic Query Set
| # | Theme | Query |
|---|---|---|
| Q1 | **Temporal Evolution** | NVDA holdings across 4 quarters of 2025 |
| Q2 | **Precision Extraction** | Pershing Square healthcare positions with CUSIPs |
| Q3 | **Cross-Fund Synthesis** | AI Infrastructure conviction: BRK vs Scion in Q4 2025 |

Each query runs **Baseline (Vanilla LLM)** vs **Enhanced (RAG)** side-by-side,
with dual-judge scoring and automated `demo_log.txt` generation.

In [1]:
# ── Set Working Directory ─────────────────────────────────────────────────────
# Ensure working directory is the Model/ folder so all relative paths
# (./data/, ./vector_db/, etc.) resolve consistently across notebooks.
import os, pathlib

_cwd = pathlib.Path.cwd()
# If VS Code opened the workspace root, navigate into Model/
if not (_cwd / "data").exists() and (_cwd / "Model" / "data").exists():
    os.chdir(_cwd / "Model")

os.makedirs("data", exist_ok=True)
print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\user\Documents\1fintech\GenAI\Individual Assignment 2\Model


In [2]:

!pip install langchain langchain-community langchain-huggingface langchain-chroma \
             sentence-transformers chromadb rank_bm25 tabulate flashrank openai \
             google-generativeai


You should consider upgrading via the 'C:\Users\user\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [ ]:

import os
import re
import json
import textwrap
import time
import pathlib
import logging
from datetime import datetime
from dataclasses import dataclass

# Suppress verbose httpx / httpcore request logs
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

import google.generativeai as genai
import openai as _openai
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from langchain_core.documents import Document
from langchain.prompts import PromptTemplate

# ── Paths ──────────────────────────────────────────────────────────────────────
CHROMA_DIR      = "./vector_db/local_chroma_storage"
PROCESSED_JSON  = "./data/processed_holdings.json"
FILINGS_BASE    = "./data/13f_filings/sec-edgar-filings"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL_NAME = "gemini-3.1-flash-lite-preview"
DEMO_LOG_PATH   = "demo_log.txt"      # written to Model folder

print("Imports OK.")


C:\Users\user\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_26316\1453962209.py:15: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See 

Imports OK.


In [4]:

# ── Gemini API setup (generator: gemini-3.1-flash-lite-preview) ────────────────
_GEMINI_KEY_FILE = pathlib.Path(__file__).parent / "gemini_api_key.txt" if "__file__" in dir() \
                   else pathlib.Path("gemini_api_key.txt")

GEMINI_API_KEY = ""
if _GEMINI_KEY_FILE.exists():
    _raw_g = _GEMINI_KEY_FILE.read_text(encoding="utf-8").strip()
    if _raw_g and _raw_g != "PASTE_YOUR_GEMINI_API_KEY_HERE":
        GEMINI_API_KEY = _raw_g

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    raise EnvironmentError(
        "Gemini API key not found. Either:\n"
        "  1. Paste your key into  Model/gemini_api_key.txt  (one line, no quotes), or\n"
        "  2. Set the environment variable: $env:GEMINI_API_KEY = 'your_key'"
    )

genai.configure(api_key=GEMINI_API_KEY)

_gemini_generator = genai.GenerativeModel(
    model_name=GENERATOR_MODEL_NAME,
    generation_config=genai.GenerationConfig(
        temperature=0.0,
        max_output_tokens=1024,
    ),
)
print(f"Gemini API key loaded. Generator: {GENERATOR_MODEL_NAME}")

JUDGE_MODEL_NAME = "gemini-2.5-pro"
JUDGE_MODEL      = JUDGE_MODEL_NAME
_gemini_judge = genai.GenerativeModel(
    model_name=JUDGE_MODEL_NAME,
    generation_config=genai.GenerationConfig(
        temperature=0.0,
        max_output_tokens=65536,
    ),
)

# ── GitHub Models endpoint (evaluator: GPT-4o-mini) ──────────────────────────
_GITHUB_TOKEN_FILE = pathlib.Path(__file__).parent / "github_token.txt" if "__file__" in dir() \
                     else pathlib.Path("github_token.txt")

GITHUB_TOKEN = ""
if _GITHUB_TOKEN_FILE.exists():
    _raw_gh = _GITHUB_TOKEN_FILE.read_text(encoding="utf-8").strip()
    if _raw_gh and _raw_gh != "PASTE_YOUR_GITHUB_TOKEN_HERE":
        GITHUB_TOKEN = _raw_gh

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

if not GITHUB_TOKEN:
    raise EnvironmentError(
        "GitHub token not found. Either:\n"
        "  1. Paste your token into  Model/github_token.txt  (one line, no quotes), or\n"
        "  2. Set the environment variable: $env:GITHUB_TOKEN = 'your_token'"
    )

_github_client = _openai.OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=GITHUB_TOKEN,
)

EVALUATOR_MODEL = "gpt-4o-mini"

# ── Rate-limit tracking ──────────────────────────────────────────────────────
_gemini_gen_call_count = 0
_evaluator_call_count = 0
_judge_call_count = 0
_GEMINI_GEN_INTER_CALL_DELAY = 2
_EVALUATOR_INTER_CALL_DELAY = 2
_JUDGE_INTER_CALL_DELAY = 2

# ── LLM-as-Judge prompt template ─────────────────────────────────────────────
LLM_JUDGE_TEMPLATE = """You are an expert financial AI evaluator assessing RAG pipeline quality.

Rate the ANSWER below on a 1-5 scale for ACCURACY, SPECIFICITY, and GROUNDING.
Use the PROVIDED CONTEXT as the ground-truth reference for 13F filing data.

CONTEXT (retrieved 13F filing chunks — treat as ground truth):
{context}

QUESTION:
{question}

ANSWER TO EVALUATE:
{answer}

SCORING RUBRIC:
5 = Excellent: answer cites specific data (fund names, share counts, CUSIPs, dates) that are
    directly confirmed by the context. Claims are precise and verifiable.
    Minor formatting differences do not reduce the score.
4 = Good: answer is mostly correct with specific citations. May have one minor inaccuracy
    or omission, but core factual claims are supported by the context.
3 = Adequate: answer is directionally correct and addresses the question, but is missing
    some key specifics that ARE available in the context, or includes partial data.
2 = Poor: answer makes specific claims that contradict the context, OR gives a generic
    response when the context contains specific data that should have been cited.
1 = Fail: answer is entirely wrong, fabricates data not in context, gives an error message,
    OR refuses to answer when the context clearly contains the relevant information.

IMPORTANT for scoring:
- Credit partial answers: if 3 of 4 data points are correct, score 4 not 2.
- A local model may phrase things differently than a cloud model — judge the factual
  content, not the prose style.
- PARTIAL CONTEXT: The context may contain data from DIFFERENT funds or periods than the
  question asks about. If the question asks about Fund X but the context only has Fund Y
  data, an answer that (a) states Fund X data is unavailable AND (b) analyses the Fund Y
  data that IS present should score 3-4. A blanket refusal like 'Insufficient data' with
  no analysis of available data should score 2.
- Fabrication check: an answer that invents specific numbers, dates, or fund names not
  present anywhere in the context should score 1, even if the claims sound plausible.
- An answer that correctly states 'data not available' when the context truly contains
  NO relevant filing data at all should score 4-5 (appropriate refusal).
- Generic/vague answers score lower than specific, data-backed answers.

Respond with ONLY a valid JSON object — no extra text before or after:
{{"score": <integer 1-5>, "rationale": "<one concise sentence>"}}"""

LLM_JUDGE_PROMPT = PromptTemplate(
    template=LLM_JUDGE_TEMPLATE,
    input_variables=["context", "question", "answer"],
)


def _build_judge_prompt(response, question, context_docs):
    """Build the judge prompt text (shared by both evaluator and judge)."""
    context_text = "\n\n---\n\n".join(
        d.page_content[:1500] for d in context_docs
    ) if context_docs else "(no context retrieved — baseline evaluation)"
    return LLM_JUDGE_PROMPT.format(
        context=context_text,
        question=question,
        answer=response[:1500],
    )


def _call_github_model(model_name, prompt_text, role_label, call_delay):
    """Call a GitHub Models endpoint and parse JSON score response."""
    _MAX_RETRIES = 3
    for _attempt in range(1, _MAX_RETRIES + 1):
        try:
            resp = _github_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": "You are an expert financial AI evaluator. Respond with ONLY valid JSON."},
                    {"role": "user", "content": prompt_text},
                ],
                max_tokens=256,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
            json_match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group())
                score = max(1, min(5, int(parsed.get("score", 3))))
                rationale = str(parsed.get("rationale", "No rationale provided."))
                return score, rationale
            raise ValueError(f"No JSON found in {model_name} response: {raw[:300]}")
        except Exception as exc:
            err_str = str(exc)
            is_rate_limit = "429" in err_str or "rate" in err_str.lower() or "quota" in err_str.lower()
            if is_rate_limit and _attempt < _MAX_RETRIES:
                wait = call_delay * _attempt * 3
                print(f"\n  [RATE LIMIT] {role_label} ({model_name}) attempt {_attempt}/{_MAX_RETRIES} "
                      f"hit rate limit. Retrying in {wait}s ...")
                time.sleep(wait)
                continue
            raise


def _call_gemini_judge(prompt_text, role_label, call_delay):
    """Call Gemini 2.5 Pro as judge via Google AI Studio and parse JSON score response."""
    _MAX_RETRIES = 3
    for _attempt in range(1, _MAX_RETRIES + 1):
        try:
            system = "You are an expert financial AI evaluator. Respond with ONLY valid JSON."
            full_prompt = f"{system}\n\n{prompt_text}"

            gen_config = genai.GenerationConfig(
                temperature=0.0,
                max_output_tokens=65536,
            )
            resp = _gemini_judge.generate_content(
                full_prompt,
                generation_config=gen_config,
            )
            # Handle empty response (thinking tokens can exhaust budget)
            raw = None
            try:
                raw = resp.text.strip()
            except (ValueError, AttributeError):
                # .text throws if no valid Part — try candidates directly
                if resp.candidates and resp.candidates[0].content.parts:
                    raw = resp.candidates[0].content.parts[0].text.strip()
            if not raw:
                _reason = getattr(resp.candidates[0], 'finish_reason', 'unknown') if resp.candidates else 'no_candidates'
                raise ValueError(f"Gemini returned empty response (finish_reason={_reason})")
            json_match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group())
                score = max(1, min(5, int(parsed.get("score", 3))))
                rationale = str(parsed.get("rationale", "No rationale provided."))
                return score, rationale
            raise ValueError(f"No JSON found in {JUDGE_MODEL} response: {raw[:300]}")
        except Exception as exc:
            err_str = str(exc)
            is_rate_limit = "429" in err_str or "RESOURCE_EXHAUSTED" in err_str
            is_empty_response = "empty response" in err_str or "finish_reason" in err_str
            if (is_rate_limit or is_empty_response) and _attempt < _MAX_RETRIES:
                wait = call_delay * _attempt * 3
                _label = "RATE LIMIT" if is_rate_limit else "EMPTY RESPONSE"
                print(f"\n  [{_label}] {role_label} ({JUDGE_MODEL}) attempt {_attempt}/{_MAX_RETRIES} "
                      f"— {err_str[:120]}. Retrying in {wait}s ...")
                time.sleep(wait)
                continue
            raise


def _score_faithfulness_keywords(response, expect_no_data, required_terms):
    """Keyword-count fallback scorer (used only when both judges fail)."""
    resp_lower = response.lower()
    if expect_no_data:
        refusal_signals = [
            "not available", "no information", "does not exist",
            "not found", "not present", "cannot find", "no position",
            "not in", "not mentioned", "no data",
        ]
        return 5 if any(s in resp_lower for s in refusal_signals) else 1
    if not required_terms:
        return 3
    hits = [t for t in required_terms if t.lower() in resp_lower]
    ratio = len(hits) / len(required_terms)
    if ratio == 1.0:   return 5
    if ratio >= 0.75:  return 4
    if ratio >= 0.5:   return 3
    if ratio > 0:      return 2
    return 1


def score_evaluator(response, question, context_docs, expect_no_data=False, required_terms=None):
    """GPT-4o-mini evaluator via GitHub Models — faithfulness scorer."""
    global _evaluator_call_count

    prompt_text = _build_judge_prompt(response, question, context_docs)

    if _evaluator_call_count > 0:
        time.sleep(_EVALUATOR_INTER_CALL_DELAY)

    try:
        score, rationale = _call_github_model(
            EVALUATOR_MODEL, prompt_text, "Evaluator", _EVALUATOR_INTER_CALL_DELAY
        )
        _evaluator_call_count += 1
        return score, rationale, "gpt4omini_evaluator"
    except Exception as exc:
        print(f"  [WARN] Evaluator ({EVALUATOR_MODEL}) failed: {str(exc)[:150]}")
        fb_score = _score_faithfulness_keywords(response, expect_no_data, required_terms or [])
        return fb_score, f"[FALLBACK-KEYWORD] Evaluator error: {exc}", "keyword_fallback"


def score_judge(response, question, context_docs, expect_no_data=False, required_terms=None):
    """Gemini 2.5 Pro judge via Google AI Studio — faithfulness scorer."""
    global _judge_call_count

    prompt_text = _build_judge_prompt(response, question, context_docs)

    if _judge_call_count > 0:
        time.sleep(_JUDGE_INTER_CALL_DELAY)

    try:
        score, rationale = _call_gemini_judge(
            prompt_text, "Judge", _JUDGE_INTER_CALL_DELAY
        )
        _judge_call_count += 1
        return score, rationale, "gemini2_judge"
    except Exception as exc:
        print(f"  [WARN] Judge ({JUDGE_MODEL}) failed: {str(exc)[:150]}")
        fb_score = _score_faithfulness_keywords(response, expect_no_data, required_terms or [])
        return fb_score, f"[FALLBACK-KEYWORD] Judge error: {exc}", "keyword_fallback"


def score_dual(response, question, context_docs, expect_no_data=False, required_terms=None):
    """Run both evaluator and judge, return averaged score."""
    e_score, e_rat, e_method = score_evaluator(response, question, context_docs, expect_no_data, required_terms)
    j_score, j_rat, j_method = score_judge(response, question, context_docs, expect_no_data, required_terms)
    avg = round((e_score + j_score) / 2, 1)
    return avg, e_score, e_rat, e_method, j_score, j_rat, j_method


def grounding_check(response, filings_base):
    """Scans source full-submission.txt files for any 9-char CUSIP in the response."""
    resp_lower = response.lower()

    refusal_phrases = [
        "not present", "not available", "no information",
        "does not exist", "not found", "insufficient data",
        "no data", "cannot find",
    ]
    is_refusal = any(p in resp_lower for p in refusal_phrases)

    # Match 9-char alphanumeric tokens that contain at least one digit
    # (real CUSIPs like 037833100 always have digits; this excludes English words
    # like FINANCIAL, BERKSHIRE, PORTFOLIO that the old regex falsely matched).
    _all_9char = set(re.findall(r'\b[A-Z0-9]{9}\b', response.upper()))
    cusips_in_response = {t for t in _all_9char if re.search(r'\d', t)}
    if not cusips_in_response:
        if is_refusal:
            return None, "Refusal/empty answer — grounding indeterminate (no CUSIPs to verify)."
        return True, "No specific CUSIPs to verify (narrative answer)."

    corpus = ""
    for root, _, files in os.walk(filings_base):
        for fname in files:
            if fname == "full-submission.txt":
                try:
                    with open(os.path.join(root, fname), "r",
                              encoding="utf-8", errors="ignore") as f:
                        corpus += f.read(5_000_000).upper()
                except OSError:
                    continue

    verified   = [c for c in cusips_in_response if c in corpus]
    unverified = [c for c in cusips_in_response if c not in corpus]

    grounded = len(unverified) == 0
    parts = []
    if verified:   parts.append(f"Verified CUSIPs: {verified}")
    if unverified: parts.append(f"UNVERIFIED (possible hallucination): {unverified}")

    return grounded, " | ".join(parts) if parts else "No verifiable identifiers found."


def _call_gemini(system_content, user_content, max_tokens=1024):
    """Call Gemini 3.1 Flash-Lite as the RAG answer generator."""
    global _gemini_gen_call_count

    if _gemini_gen_call_count > 0:
        time.sleep(_GEMINI_GEN_INTER_CALL_DELAY)

    prompt = f"{system_content}\n\n{user_content}"

    for _attempt in range(1, 4):
        try:
            gen_config = genai.GenerationConfig(
                temperature=0.0,
                max_output_tokens=max_tokens,
            )
            resp = _gemini_generator.generate_content(
                prompt,
                generation_config=gen_config,
            )
            _gemini_gen_call_count += 1
            return resp.text.strip()
        except Exception as exc:
            err_str = str(exc)
            is_rate_limit = "429" in err_str or "RESOURCE_EXHAUSTED" in err_str
            if _attempt < 3:
                wait = 5 * _attempt if not is_rate_limit else 10 * _attempt
                print(f"    [GEMINI GEN RETRY] attempt {_attempt}/3: {err_str[:100]}. "
                      f"Retrying in {wait}s ...")
                time.sleep(wait)
                continue
            return f"[GEMINI ERROR] {exc}"

# ── Initialise embeddings and vector store ────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
)

# ── Rebuild BM25 + EnsembleRetriever + FlashrankRerank (dual) ─────────────────
print("Rebuilding BM25 retriever from source JSON ...")
with open(PROCESSED_JSON, "r", encoding="utf-8") as f:
    _raw_docs = json.load(f)
_documents = [
    Document(page_content=e["text"], metadata=e["metadata"])
    for e in _raw_docs
]

bm25_retriever = BM25Retriever.from_documents(_documents)
bm25_retriever.k = 10

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)

try:
    compressor_narrow = FlashrankRerank(top_n=8)
    compressor_wide   = FlashrankRerank(top_n=20)
    reranking_retriever = ContextualCompressionRetriever(
        base_compressor=compressor_narrow,
        base_retriever=hybrid_retriever,
    )
    reranking_retriever_wide = ContextualCompressionRetriever(
        base_compressor=compressor_wide,
        base_retriever=hybrid_retriever,
    )
    USE_RERANKER = True
    print("FlashrankRerank       : ACTIVE (narrow=8, wide=20)")
except Exception as _exc:
    reranking_retriever = hybrid_retriever
    reranking_retriever_wide = hybrid_retriever
    USE_RERANKER = False
    print(f"[WARN] FlashrankRerank unavailable ({_exc}); using EnsembleRetriever only")

RETRIEVAL_LABEL = "Hybrid BM25+Vector + FlashrankRerank" if USE_RERANKER else "Hybrid BM25+Vector"

# ── Entity-scoped filter ──────────────────────────────────────────────────────
_FUND_ALIAS_MAP = {}
_all_fund_names = sorted(set(
    d.metadata.get("fund_name", "") for d in _documents if d.metadata.get("fund_name")
))
for _fn in _all_fund_names:
    _fn_lower = _fn.lower()
    _FUND_ALIAS_MAP[_fn_lower] = _fn
    if "(" in _fn:
        _base = _fn[:_fn.index("(")].strip()
        _FUND_ALIAS_MAP[_base.lower()] = _fn
        _paren = _fn[_fn.index("(") + 1 : _fn.index(")")].strip()
        if len(_paren) > 3:
            _FUND_ALIAS_MAP[_paren.lower()] = _fn
    _cleaned = _fn.split("(")[0] if "(" in _fn else _fn
    for _suffix in ["Inc", "LLC", "Corp", "Ltd", "PLC", "AG", "N.A.", "Co"]:
        _cleaned = _cleaned.replace(_suffix, "")
    _words = _cleaned.strip().split()
    if len(_words) >= 2:
        _two_word = " ".join(_words[:2]).lower().strip()
        if len(_two_word) > 5:
            _FUND_ALIAS_MAP[_two_word] = _fn

def _extract_fund_entities(question):
    q_lower = question.lower()
    _BROAD_SIGNALS = ["all funds", "which funds", "across all", "all available", "any funds"]
    if any(sig in q_lower for sig in _BROAD_SIGNALS):
        return set()
    matched = set()
    for alias in sorted(_FUND_ALIAS_MAP.keys(), key=len, reverse=True):
        if len(alias) <= 4:
            continue
        if alias in q_lower:
            matched.add(_FUND_ALIAS_MAP[alias])
    return matched

def _entity_filter(docs, target_funds):
    if not target_funds:
        return docs
    filtered = [d for d in docs if d.metadata.get("fund_name", "") in target_funds]
    return filtered if filtered else docs

# ── Temporal filter ───────────────────────────────────────────────────────────
def _temporal_filter(docs, question):
    q_lower = question.lower()
    PERIOD_MAP = {
        ("feb 2026", "february 2026", "q4 2025"): ("2025-12-31", "2026-01-01"),
        ("nov 2025", "november 2025", "q3 2025"): ("2025-09-30", "2025-10-01"),
        ("aug 2025", "august 2025", "q2 2025"):   ("2025-06-30", "2025-07-01"),
        ("may 2025", "q1 2025"):                   ("2025-03-31", "2025-04-01"),
    }
    target_period = None
    min_filing = None
    for signals, (period, filing_min) in PERIOD_MAP.items():
        if any(s in q_lower for s in signals):
            target_period = period
            min_filing = filing_min
            break

    # Fallback: "latest", "most recent", "current" → use the newest period in docs
    if target_period is None:
        _LATEST_SIGNALS = ["latest", "most recent", "current", "newest"]
        if any(s in q_lower for s in _LATEST_SIGNALS):
            periods = [d.metadata.get("period_of_report", "") for d in docs
                       if d.metadata.get("period_of_report", "") not in ("", "UNKNOWN")]
            if periods:
                target_period = max(periods)

    if target_period is None:
        return docs
    filtered = [
        d for d in docs
        if d.metadata.get("period_of_report", "") == target_period
        or (d.metadata.get("period_of_report", "") in ("", "UNKNOWN")
            and min_filing and d.metadata.get("filing_date", "") >= min_filing)
    ]
    return filtered if filtered else docs

# ── Fund diversity filter ─────────────────────────────────────────────────────
def _diversify_by_fund(docs, max_per_fund=2):
    fund_counts = {}
    diverse = []
    for d in docs:
        fund = d.metadata.get("fund_name", "")
        fund_counts[fund] = fund_counts.get(fund, 0) + 1
        if fund_counts[fund] <= max_per_fund:
            diverse.append(d)
    return diverse if diverse else docs

# ── Chunk pre-filter ──────────────────────────────────────────────────────────
def _prefilter_chunks(question, docs):
    keywords = [w.lower() for w in question.split() if len(w) > 3]
    filtered = []
    for d in docs:
        lines = d.page_content.split("\n")
        header_lines = []
        data_lines = []
        for line in lines:
            stripped = line.strip()
            if stripped.startswith("|"):
                if "nameOfIssuer" in line or ":-" in line or "--" in line:
                    header_lines.append(line)
                else:
                    data_lines.append(line)
            else:
                header_lines.append(line)
        matched_rows = [r for r in data_lines if any(k in r.lower() for k in keywords)]
        if matched_rows:
            filtered.append("\n".join(header_lines + matched_rows))
        else:
            filtered.append(d.page_content)
    return filtered

# ── Chain routing: Stuff for exact-lookup, Map-Reduce for synthesis ───────────
MAP_REDUCE_TYPES = {
    "Cross-Fund Sector Synthesis",
    "Comparative / Deep Context",
    "Sector Concentration",
}

def run_stuff_chain(question, docs):
    filtered_chunks = _prefilter_chunks(question, docs)
    context = "\n\n---\n\n".join(filtered_chunks)
    system = (
        "You are a precise financial analyst. "
        "Use ONLY the 13F filing data provided below — no outside knowledge. "
        "Extract data directly from the tables and metadata headers. "
        "If the specific data point is absent from the provided excerpts, "
        "state: 'This information is not present in the retrieved filing data.' "
        "However, if the data is present for DIFFERENT funds or periods than asked about, "
        "report what IS available and note which requested entities are missing. "
        "CRITICAL: Copy ALL numerical values, share counts, CUSIPs, dates, and fund names "
        "VERBATIM from the context. Do not paraphrase, round, or restate them. "
        "You MUST cite the exact fund name as it appears in the context header (e.g., "
        "'Fund: Treasurer of the State of North Carolina') and the accession number "
        "for every claim. Never use generic labels like 'Fund 1' or 'the fund'."
    )
    user = (
        f"Filing data:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer using ONLY the data above. "
        "Cite exact fund name, accession number, share count, CUSIP, and filing date:"
    )
    return _call_gemini(system, user, max_tokens=1024)

def run_map_reduce_chain(question, docs):
    filtered_chunks = _prefilter_chunks(question, docs)
    map_system = (
        "You are a financial analyst reviewing a single 13F filing excerpt. "
        "First, extract the fund name, filing date, period of report, and accession "
        "number from the metadata header. Then extract the holdings data most relevant "
        "to the question — include issuer names, CUSIPs, share counts, market values, "
        "and any sector/industry identifiers present. Copy ALL numerical values and "
        "identifiers VERBATIM. Always extract what IS present — even if the excerpt "
        "is from a different fund than the one asked about. "
        "If the excerpt is completely empty or contains no tabular data, "
        "state: 'No holdings data in this excerpt.'"
    )
    summaries = []
    for i, chunk_text in enumerate(filtered_chunks):
        map_user = (
            f"Context (excerpt {i+1} of {len(filtered_chunks)}):\n{chunk_text}\n\n"
            f"Question: {question}\n\n"
            "Extract: fund name, accession number, filing date, period of report, "
            "and all holdings data relevant to the question above "
            "(issuer names, CUSIPs, values, share counts):"
        )
        summary = _call_gemini(map_system, map_user, max_tokens=400)
        summaries.append(summary)

    reduce_system = (
        "You are a meticulous financial analyst consolidating multiple 13F filing excerpts. "
        "Use ONLY the summaries below — no outside knowledge. "
        "Preserve all numerical values, CUSIPs, share counts, dates, and fund names "
        "EXACTLY as stated. Do not invent, round, or paraphrase figures. "
        "IMPORTANT: If the question asks about specific funds that are NOT present in "
        "the summaries, state which funds were requested but unavailable, then analyse "
        "the data that IS available. Identify trends: compare values across quarters "
        "if multiple periods are present, note which positions grew or shrank. "
        "Always cite the exact fund name and accession number for every claim."
    )
    joined = "\n\n---\n\n".join(f"Excerpt {i+1}:\n{s}" for i, s in enumerate(summaries))
    reduce_user = (
        f"Summaries from {len(filtered_chunks)} retrieved chunks:\n{joined}\n\n"
        f"Question: {question}\n\n"
        "Final consolidated answer (cite exact fund names, accession numbers, share counts, "
        "CUSIPs, and filing dates for every claim. If multiple quarters are present, "
        "compare values across periods to identify trends):"
    )

    return _call_gemini(reduce_system, reduce_user, max_tokens=1024)

print("Pipeline ready.")
print(f"Filters   : entity + temporal + diversity")
print("Architecture:")
print(f"Retrieval : {RETRIEVAL_LABEL}")
print(f"BM25      : {len(_documents)} docs, k=10")
print(f"Store     : {vectorstore._collection.count()} vectors")
print(f"  Generator  : {GENERATOR_MODEL_NAME} via Google AI Studio (Google DeepMind)")
print(f"  Evaluator  : {EVALUATOR_MODEL} via GitHub Models (OpenAI)")
print(f"  Judge      : {JUDGE_MODEL} via Google AI Studio (Google DeepMind)")
print(f"  → Two different providers (Google / OpenAI) — no self-grading bias.")

Gemini API key loaded. Generator: gemini-3.1-flash-lite-preview


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10302.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Rebuilding BM25 retriever from source JSON ...
FlashrankRerank       : ACTIVE (narrow=8, wide=20)
Pipeline ready.
Filters   : entity + temporal + diversity
Architecture:
Retrieval : Hybrid BM25+Vector + FlashrankRerank
BM25      : 95377 docs, k=10
Store     : 95377 vectors
  Generator  : gemini-3.1-flash-lite-preview via Google AI Studio (Google DeepMind)
  Evaluator  : gpt-4o-mini via GitHub Models (OpenAI)
  Judge      : gemini-2.5-pro via Google AI Studio (Google DeepMind)
  → Two different providers (Google / OpenAI) — no self-grading bias.


In [5]:
# ── 3 Thematic Demo Queries ────────────────────────────────────────────────────
# Designed for maximum marker impact across three RAG challenge dimensions:
#   Q1 (Temporal):   Multi-quarter evolution — requires ALL 4 periods, Map-Reduce
#   Q2 (Extraction): Single-fund precision with CUSIPs — Stuff chain
#   Q3 (Synthesis):  Cross-fund sector comparison — Map-Reduce

@dataclass
class DemoQuery:
    label: str
    query_type: str          # drives chain routing (Stuff vs Map-Reduce)
    theme: str               # human-readable theme label
    question: str
    commentary: str = ""     # populated dynamically after retrieval
    top_k: int = 6

DEMO_QUERIES = [
    DemoQuery(
        label="Q1",
        query_type="Comparative / Deep Context",       # → Map-Reduce
        theme="Temporal Evolution",
        question=(
            "Analyze the evolution of Nvidia (NVDA) holdings across the four "
            "reported quarters of 2025."
        ),
        top_k=12,
    ),
    DemoQuery(
        label="Q2",
        query_type="Numerical Extraction from Table",  # → Stuff
        theme="Precision Extraction",
        question=(
            "Identify the top three high-conviction healthcare positions for "
            "Pershing Square in the February 2026 filings with CUSIPs."
        ),
        top_k=6,
    ),
    DemoQuery(
        label="Q3",
        query_type="Comparative / Deep Context",       # → Map-Reduce
        theme="Cross-Fund Synthesis",
        question=(
            "Compare 'AI Infrastructure' conviction between Berkshire Hathaway "
            "and Scion Asset Management in Q4 2025."
        ),
        top_k=10,
    ),
]

print(f"Demo queries defined: {len(DEMO_QUERIES)}")
for dq in DEMO_QUERIES:
    chain = "Map-Reduce" if dq.query_type in MAP_REDUCE_TYPES else "Stuff"
    print(f"  {dq.label} [{dq.theme}] → {chain} chain, top_k={dq.top_k}")


Demo queries defined: 3
  Q1 [Temporal Evolution] → Map-Reduce chain, top_k=12
  Q2 [Precision Extraction] → Stuff chain, top_k=6
  Q3 [Cross-Fund Synthesis] → Map-Reduce chain, top_k=10


In [6]:

# ── Run Baseline vs Enhanced RAG for each thematic query ────────────────────────
# Reset session counters
_gemini_gen_call_count = 0
_evaluator_call_count = 0
_judge_call_count = 0

SEP   = "=" * 72
THIN  = "-" * 72

demo_results = []

for dq in DEMO_QUERIES:
    print(f"\n{SEP}")
    print(f"  {dq.label} | {dq.theme} | {dq.query_type}")
    print(f"{SEP}")
    print(f"  Q: {dq.question}")
    print(THIN)

    # ────────────────── BASELINE (Vanilla LLM, no context) ──────────────────
    print(f"  [1/2] BASELINE ({GENERATOR_MODEL_NAME}, zero context) ...")
    baseline_system = (
        "You are a financial analyst. Answer the following question about "
        "institutional 13F-HR SEC filings. If you do not have the specific data, "
        "state that clearly rather than guessing."
    )
    baseline_resp = _call_gemini(baseline_system, dq.question)

    # ────────────────── ENHANCED RAG (full pipeline) ────────────────────────
    print(f"  [2/2] ENHANCED RAG ({RETRIEVAL_LABEL}) ...")

    # Select narrow or wide reranker based on query scope
    target_funds = _extract_fund_entities(dq.question)
    is_broad = len(target_funds) == 0
    _retriever = reranking_retriever_wide if is_broad else reranking_retriever

    try:
        docs = _retriever.invoke(dq.question)
    except Exception as exc:
        print(f"  [WARN] Reranker failed ({exc}); falling back to hybrid retriever")
        docs = hybrid_retriever.invoke(dq.question)[:8]

    # Entity filter
    if target_funds:
        docs = _entity_filter(docs, target_funds)
        print(f"    Entity filter  : {sorted(target_funds)} → {len(docs)} chunks")

    # Temporal filter
    _pre = len(docs)
    docs = _temporal_filter(docs, dq.question)
    if len(docs) != _pre:
        print(f"    Temporal filter: {len(docs)}/{_pre} chunks retained")

    # Diversity filter for broad queries
    if is_broad:
        _pre = len(docs)
        docs = _diversify_by_fund(docs, max_per_fund=2)
        if len(docs) != _pre:
            print(f"    Diversity filter: {len(docs)}/{_pre} chunks retained")

    # Route to Stuff or Map-Reduce
    chain_label = "Map-Reduce" if dq.query_type in MAP_REDUCE_TYPES else "Stuff"
    print(f"    Chain: {chain_label}  |  Chunks: {len(docs)}")

    if dq.query_type in MAP_REDUCE_TYPES:
        enhanced_resp = run_map_reduce_chain(dq.question, docs)
    else:
        enhanced_resp = run_stuff_chain(dq.question, docs)

    # ── Dual-judge scoring ────────────────────────────────────────────────
    _expect_no_data = "not present in any filing" in dq.question.lower() or \
                      "does not exist" in dq.question.lower()

    print(f"    Scoring BASELINE ...")
    b_avg, b_eval_s, b_eval_r, b_eval_m, b_judge_s, b_judge_r, b_judge_m = \
        score_dual(baseline_resp, dq.question, docs, expect_no_data=_expect_no_data)
    b_grounded, b_grounding_detail = grounding_check(baseline_resp, FILINGS_BASE)

    print(f"    Scoring ENHANCED ...")
    r_avg, r_eval_s, r_eval_r, r_eval_m, r_judge_s, r_judge_r, r_judge_m = \
        score_dual(enhanced_resp, dq.question, docs, expect_no_data=_expect_no_data)
    r_grounded, r_grounding_detail = grounding_check(enhanced_resp, FILINGS_BASE)

    grounded_label = {True: "PASSED", False: "FAILED", None: "N/A"}[r_grounded]

    # ── Build source list ─────────────────────────────────────────────────
    sources = []
    seen = set()
    for doc in docs:
        m = doc.metadata
        key = (m["fund_name"], m["chunk_index"])
        if key not in seen:
            seen.add(key)
            snippet = doc.page_content[:300].replace("\n", " ")
            sources.append({
                "fund"      : m["fund_name"],
                "date"      : m["filing_date"],
                "period"    : m.get("period_of_report", "unknown"),
                "accession" : m.get("accession_number", "unknown"),
                "chunk"     : f"{m['chunk_index']}/{m['total_chunks']}",
                "score"     : round(float(m.get("relevance_score", -1.0)), 4),
                "snippet"   : snippet,
            })

    # ── Auto-generate technical commentary: Retrieval Strategy + Grounding Proof ──
    fund_names = sorted(set(s["fund"] for s in sources))
    fund_list = ", ".join(fund_names[:5]) or "no funds"
    all_accessions = sorted(set(s["accession"] for s in sources if s["accession"] != "unknown"))
    primary_accession = all_accessions[0] if all_accessions else "N/A"
    periods_in_docs = sorted(set(s["period"] for s in sources if s["period"] not in ("", "unknown")))
    num_unique_funds = len(set(s["fund"] for s in sources))

    # — Retrieval Strategy commentary —
    retrieval_parts = []
    retrieval_parts.append(
        f"Hybrid search (BM25 lexical + cosine-similarity vector) retrieved initial "
        f"candidates. BM25 resolved keyword matches (e.g. ticker names, fund names) "
        f"while the vector arm captured semantic intent."
    )
    if USE_RERANKER:
        _rerank_mode = "wide (top_n=20)" if is_broad else "narrow (top_n=8)"
        retrieval_parts.append(
            f"FlashrankRerank ({_rerank_mode}) re-scored candidates by cross-encoder "
            f"relevance, promoting the most query-aligned chunks."
        )
    if target_funds:
        retrieval_parts.append(
            f"Entity filter scoped results to: {sorted(target_funds)}."
        )
    if periods_in_docs:
        retrieval_parts.append(
            f"Temporal filter isolated period(s) {', '.join(periods_in_docs)}, "
            f"preventing cross-quarter bleeding."
        )
    if is_broad and num_unique_funds > 1:
        retrieval_parts.append(
            f"Fund-diversity filter ensured representation across {num_unique_funds} funds."
        )
    retrieval_parts.append(
        f"Final context: {len(sources)} chunk(s) from {fund_list} "
        f"routed through the {chain_label} synthesis chain."
    )
    retrieval_strategy = " ".join(retrieval_parts)

    # — Grounding Proof commentary —
    grounding_parts = []
    if all_accessions:
        grounding_parts.append(
            f"Accession Numbers cited: {', '.join(all_accessions[:6])}."
        )
    else:
        grounding_parts.append("No accession numbers recovered from retrieved chunks.")
    grounding_parts.append(
        f"CUSIP grounding check: {grounded_label}. {r_grounding_detail}"
    )
    if periods_in_docs:
        grounding_parts.append(
            f"Period(s) of report in evidence: {', '.join(periods_in_docs)}."
        )
    grounding_proof = " ".join(grounding_parts)

    # — Private Data Gap context —
    data_gap = (
        f"The vanilla LLM's training data does not include 2025-2026 13F-HR filings. "
        f"The RAG pipeline bridged this 'Private Data Gap' by retrieving {len(sources)} "
        f"chunk(s) from {fund_list}, injecting actual SEC filing tables into the prompt "
        f"so the generator could cite exact share counts, CUSIPs, and market values."
    )

    # — Synthesis commentary —
    if chain_label == "Map-Reduce":
        synthesis = (
            f"Map-Reduce chain processed {len(sources)} chunks across "
            f"{num_unique_funds} distinct fund(s). Each Map call extracted holdings "
            f"data verbatim; the Reduce step consolidated cross-fund/cross-quarter "
            f"evidence into a unified analytical response."
        )
    else:
        synthesis = (
            f"Stuff chain concatenated {len(sources)} chunk(s) from "
            f"{num_unique_funds} fund(s) into a single prompt — sufficient for "
            f"this focused extraction query."
        )

    dq._tech_commentary = {
        "retrieval_strategy": retrieval_strategy,
        "grounding_proof"   : grounding_proof,
        "data_gap"          : data_gap,
        "synthesis"         : synthesis,
    }

    demo_results.append({
        "dq"                 : dq,
        "baseline"           : baseline_resp,
        "enhanced"           : enhanced_resp,
        "sources"            : sources,
        "retrieval"          : RETRIEVAL_LABEL,
        "chain_type"         : chain_label,
        # Averaged scores
        "baseline_score"     : b_avg,
        "enhanced_score"     : r_avg,
        "delta"              : round(r_avg - b_avg, 1),
        # Evaluator (GPT-4o-mini)
        "b_eval_score"       : b_eval_s,
        "b_eval_rationale"   : b_eval_r,
        "b_eval_method"      : b_eval_m,
        "r_eval_score"       : r_eval_s,
        "r_eval_rationale"   : r_eval_r,
        "r_eval_method"      : r_eval_m,
        # Judge (Gemini 2.5 Pro)
        "b_judge_score"      : b_judge_s,
        "b_judge_rationale"  : b_judge_r,
        "b_judge_method"     : b_judge_m,
        "r_judge_score"      : r_judge_s,
        "r_judge_rationale"  : r_judge_r,
        "r_judge_method"     : r_judge_m,
        # Grounding
        "b_grounding_detail" : b_grounding_detail,
        "r_grounding_detail" : r_grounding_detail,
        "enhanced_grounded"  : r_grounded,
    })

    # ── Inline side-by-side summary ──────────────────────────────────────
    print(f"\n  {'BASELINE':^36} {'ENHANCED (RAG)':^36}")
    print(f"  {'─'*36} {'─'*36}")
    b_preview = baseline_resp[:200].replace('\n', ' ')
    r_preview = enhanced_resp[:200].replace('\n', ' ')
    print(f"  {b_preview[:36]:36} {r_preview[:36]:36}")
    print(f"  Score: {b_avg}/5                      Score: {r_avg}/5  (Δ {r_avg - b_avg:+.1f})")
    print(f"  Grounded: {'N/A':10}                  Grounded: {grounded_label}")
    print(f"  Sources: {[s['fund'][:25] for s in sources[:3]]}")

print(f"\n{SEP}")
print(f"  All {len(demo_results)} thematic queries complete.")
print(f"  Evaluator calls ({EVALUATOR_MODEL}): {_evaluator_call_count}")
print(f"  Judge calls ({JUDGE_MODEL}): {_judge_call_count}")
print(SEP)



  Q1 | Temporal Evolution | Comparative / Deep Context
  Q: Analyze the evolution of Nvidia (NVDA) holdings across the four reported quarters of 2025.
------------------------------------------------------------------------
  [1/2] BASELINE (gemini-3.1-flash-lite-preview, zero context) ...
  [2/2] ENHANCED RAG (Hybrid BM25+Vector + FlashrankRerank) ...
    Diversity filter: 7/20 chunks retained
    Chain: Map-Reduce  |  Chunks: 7
    Scoring BASELINE ...
    Scoring ENHANCED ...
  [WARN] Judge (gemini-2.5-pro) failed: Expecting ',' delimiter: line 5 column 4 (char 273)

                BASELINE                          ENHANCED (RAG)           
  ──────────────────────────────────── ────────────────────────────────────
  As a financial analyst, I must first Based on the provided 13F filing exc
  Score: 1.5/5                      Score: 3.5/5  (Δ +2.0)
  Grounded: N/A                         Grounded: PASSED
  Sources: ['Treasurer of the State of', 'Treasurer of the State of', 'BNY Mel

In [7]:

# ── Pretty-print side-by-side demo output ─────────────────────────────────────
WRAP = 100
SEP  = "=" * 72
THIN = "-" * 72

def format_demo_block(r: dict) -> str:
    """Format a single demo result into the structured log entry with
    Retrieval Strategy + Grounding Proof commentary."""
    dq        = r["dq"]
    retrieval = r.get("retrieval", "unknown")
    chain_type = r.get("chain_type", "Stuff")
    tc       = getattr(dq, "_tech_commentary", {})
    theme     = getattr(dq, "theme", dq.query_type)

    lines = []
    lines.append(SEP)
    lines.append(f"  {dq.label} — {theme}")
    lines.append(SEP)

    # ── SAMPLE INPUT ──────────────────────────────────────────────────
    lines.append("")
    lines.append("[SAMPLE INPUT]")
    lines.append(f"  Query : {dq.question}")
    lines.append(f"  Type  : {dq.query_type}")
    lines.append(f"  Chain : {chain_type}  |  Retrieval: {retrieval}")
    lines.append("")

    # ── SYSTEM OUTPUT: Side-by-Side ───────────────────────────────────
    lines.append("[SYSTEM OUTPUT]")
    lines.append("")

    # Baseline
    lines.append(f"  ┌─ BASELINE (Vanilla LLM — {GENERATOR_MODEL_NAME}, no context)")
    lines.append(f"  │")
    for wrap_line in textwrap.fill(r["baseline"].strip(), width=WRAP-6).split("\n"):
        lines.append(f"  │  {wrap_line}")
    lines.append(f"  │")
    lines.append(f"  │  Score: {r['baseline_score']}/5  "
                 f"[Evaluator ({EVALUATOR_MODEL}): {r['b_eval_score']}/5 | "
                 f"Judge ({JUDGE_MODEL}): {r['b_judge_score']}/5]")
    lines.append(f"  │  Grounding: {r['b_grounding_detail']}")
    lines.append(f"  └{'─' * 68}")
    lines.append("")

    # Enhanced
    _delta_str = f"Δ {r['delta']:+.1f}"
    _grounded = {True: "PASSED", False: "FAILED", None: "N/A"}[r.get("enhanced_grounded")]
    lines.append(f"  ┌─ ENHANCED (Temporal RAG — {chain_type} chain)")
    lines.append(f"  │")
    for wrap_line in textwrap.fill(r["enhanced"].strip(), width=WRAP-6).split("\n"):
        lines.append(f"  │  {wrap_line}")
    lines.append(f"  │")
    lines.append(f"  │  Score: {r['enhanced_score']}/5  ({_delta_str})  "
                 f"[Evaluator ({EVALUATOR_MODEL}): {r['r_eval_score']}/5 | "
                 f"Judge ({JUDGE_MODEL}): {r['r_judge_score']}/5]")
    lines.append(f"  │  Grounding: {_grounded} — {r['r_grounding_detail']}")
    lines.append(f"  └{'─' * 68}")
    lines.append("")

    # ── Retrieved Sources ─────────────────────────────────────────────
    lines.append(f"[RETRIEVED SOURCES] ({len(r['sources'])} chunks via {retrieval})")
    for i, s in enumerate(r["sources"], 1):
        score_str = f"  Rerank: {s['score']}" if s.get("score", -1) >= 0 else ""
        lines.append(f"  [{i}] {s['fund']}")
        lines.append(f"      Period: {s.get('period','?')}  |  Accession: {s['accession']}  "
                     f"|  Filed: {s['date']}{score_str}")
    lines.append("")

    # ── TECHNICAL COMMENTARY ──────────────────────────────────────────
    lines.append("[TECHNICAL COMMENTARY]")
    lines.append("")
    lines.append("  1. Retrieval Strategy:")
    lines.append(textwrap.fill(tc.get("retrieval_strategy", "N/A"),
                               width=WRAP, initial_indent="     ", subsequent_indent="     "))
    lines.append("")
    lines.append("  2. Grounding Proof:")
    lines.append(textwrap.fill(tc.get("grounding_proof", "N/A"),
                               width=WRAP, initial_indent="     ", subsequent_indent="     "))
    lines.append("")
    lines.append("  3. Private Data Gap:")
    lines.append(textwrap.fill(tc.get("data_gap", "N/A"),
                               width=WRAP, initial_indent="     ", subsequent_indent="     "))
    lines.append("")
    lines.append("  4. Synthesis Quality:")
    lines.append(textwrap.fill(tc.get("synthesis", "N/A"),
                               width=WRAP, initial_indent="     ", subsequent_indent="     "))
    lines.append("")
    lines.append(THIN)
    lines.append("")
    return "\n".join(lines)


# ── Print all results ─────────────────────────────────────────────────────────
for r in demo_results:
    print(format_demo_block(r))

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'SCORING SUMMARY':^72}")
print(THIN)
print(f"  {'Query':<6} {'Theme':<25} {'Baseline':>8} {'Enhanced':>8} {'Delta':>7} {'Grounded':>10}")
print(f"  {'─'*6} {'─'*25} {'─'*8} {'─'*8} {'─'*7} {'─'*10}")
for r in demo_results:
    dq = r["dq"]
    theme = getattr(dq, "theme", dq.query_type)[:25]
    _g = {True: "YES", False: "NO", None: "N/A"}[r.get("enhanced_grounded")]
    print(f"  {dq.label:<6} {theme:<25} {r['baseline_score']:>7}/5 {r['enhanced_score']:>7}/5 "
          f"{r['delta']:>+6.1f} {_g:>10}")
avg_b = round(sum(r["baseline_score"] for r in demo_results) / len(demo_results), 1)
avg_e = round(sum(r["enhanced_score"] for r in demo_results) / len(demo_results), 1)
print(f"  {'─'*6} {'─'*25} {'─'*8} {'─'*8} {'─'*7}")
print(f"  {'AVG':<6} {'':25} {avg_b:>7}/5 {avg_e:>7}/5 {avg_e - avg_b:>+6.1f}")
print(THIN)


  Q1 — Temporal Evolution

[SAMPLE INPUT]
  Query : Analyze the evolution of Nvidia (NVDA) holdings across the four reported quarters of 2025.
  Type  : Comparative / Deep Context
  Chain : Map-Reduce  |  Retrieval: Hybrid BM25+Vector + FlashrankRerank

[SYSTEM OUTPUT]

  ┌─ BASELINE (Vanilla LLM — gemini-3.1-flash-lite-preview, no context)
  │
  │  As a financial analyst, I must first clarify a critical regulatory distinction regarding SEC
  │  13F-HR filings: **The 2025 calendar year has not yet concluded, and therefore, four quarters
  │  of 2025 data do not exist.**  13F filings are submitted by institutional investment managers
  │  with at least $100 million in qualifying assets. These filings are submitted on a quarterly
  │  basis, 45 days after the end of each calendar quarter.   As of today, the most recent
  │  completed filing period is **Q3 2024** (period ending September 30, 2024). The filings for Q4
  │  2024 are due in mid-February 2025. Consequently, there is no data a

In [8]:

# ── Export demo_log.txt ───────────────────────────────────────────────────────
header = [
    "=" * 72,
    "  13F-INSIGHT RAG PIPELINE — DEMO LOG",
    "=" * 72,
    "",
    f"Generated  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Generator  : {GENERATOR_MODEL_NAME} (via Google AI Studio)",
    f"Evaluator  : {EVALUATOR_MODEL} (via GitHub Models / OpenAI)",
    f"Judge      : {JUDGE_MODEL} (via Google AI Studio / Google DeepMind)",
    f"Embedding  : {EMBEDDING_MODEL}",
    f"Retrieval  : {RETRIEVAL_LABEL}",
    f"Chain      : Routed Stuff / Map-Reduce",
    f"Filters    : entity-scoped + temporal (period_of_report) + fund diversity",
    f"Queries    : {len(demo_results)}",
    "",
    "Architecture: EnsembleRetriever (BM25 + Vector) → FlashrankRerank → Stuff/Map-Reduce",
    "",
    "Each log entry contains:",
    "  [SAMPLE INPUT]          — User query with routing metadata",
    "  [SYSTEM OUTPUT]         — Baseline vs Enhanced side-by-side with dual-judge scores",
    "  [RETRIEVED SOURCES]     — Chunks used with accession numbers",
    "  [TECHNICAL COMMENTARY]  — Retrieval Strategy, Grounding Proof,",
    "                            Private Data Gap, Synthesis Quality",
    "",
    "=" * 72,
    "",
]

log_content = "\n".join(header)
for r in demo_results:
    log_content += format_demo_block(r) + "\n"

# Append scoring summary
log_content += f"\n{'SCORING SUMMARY':^72}\n"
log_content += THIN + "\n"
log_content += f"  {'Query':<6} {'Theme':<25} {'Baseline':>8} {'Enhanced':>8} {'Delta':>7} {'Grounded':>10}\n"
log_content += f"  {'─'*6} {'─'*25} {'─'*8} {'─'*8} {'─'*7} {'─'*10}\n"
for r in demo_results:
    dq = r["dq"]
    theme = getattr(dq, "theme", dq.query_type)[:25]
    _g = {True: "YES", False: "NO", None: "N/A"}[r.get("enhanced_grounded")]
    log_content += (f"  {dq.label:<6} {theme:<25} {r['baseline_score']:>7}/5 "
                    f"{r['enhanced_score']:>7}/5 {r['delta']:>+6.1f} {_g:>10}\n")
avg_b = round(sum(r["baseline_score"] for r in demo_results) / len(demo_results), 1)
avg_e = round(sum(r["enhanced_score"] for r in demo_results) / len(demo_results), 1)
log_content += f"  {'─'*6} {'─'*25} {'─'*8} {'─'*8} {'─'*7}\n"
log_content += f"  {'AVG':<6} {'':25} {avg_b:>7}/5 {avg_e:>7}/5 {avg_e - avg_b:>+6.1f}\n"
log_content += THIN + "\n"

os.makedirs(os.path.dirname(os.path.abspath(DEMO_LOG_PATH)), exist_ok=True)
with open(DEMO_LOG_PATH, "w", encoding="utf-8") as f:
    f.write(log_content)

abs_path = os.path.abspath(DEMO_LOG_PATH)
print(f"Demo log written to : {abs_path}")
print(f"File size           : {os.path.getsize(abs_path):,} bytes")
print(f"Queries logged      : {len(demo_results)}")
print(f"Evaluator calls     : {_evaluator_call_count}")
print(f"Judge calls         : {_judge_call_count}")


Demo log written to : c:\Users\user\Documents\1fintech\GenAI\Individual Assignment 2\demo_log.txt
File size           : 24,707 bytes
Queries logged      : 3
Evaluator calls     : 6
Judge calls         : 4
